# UI Flow Embeddings — Debug & Validation Notebook
Interactive notebook for loading embeddings, aligning by `screen_key`, validating fusion, kNN sanity checks, clustering, and near-duplicate scanning. Uses core logic from `src/uiflow/`.


In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

In [9]:

# 0) Setup: import from src/uiflow
import os, sys
from pathlib import Path

REPO_ROOT = Path.cwd()  # set to your repo root
SRC_DIR = REPO_ROOT / "../.."
assert SRC_DIR.exists(), f"Cannot find src/ at {SRC_DIR}. Update REPO_ROOT."

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Repo root:", REPO_ROOT)
print("Using src:", SRC_DIR)


Repo root: c:\Users\atsumilab\Desktop\PHD\rico_dataset\paper2\CLIP\src\notebooks
Using src: c:\Users\atsumilab\Desktop\PHD\rico_dataset\paper2\CLIP\src\notebooks\..\..


In [10]:

# 1) Imports
import numpy as np
import pandas as pd

from uiflow.io.load import (
    read_meta_tsv, read_embeddings_npy, attach_screen_key, align_by_key, l2_normalize
)
from uiflow.repr.fuse import concat_fuse
from uiflow.analysis.neighbors import topk_cosine_neighbors_blockwise
from uiflow.analysis.clustering import kmeans_cluster


## Configure paths
Prefer meta TSVs that do **not** contain `e0..` columns. If your CLIP TSV includes them, we ignore them.


In [ ]:

# 2) Paths (edit these)
DATA_PATH = Path("../../processed_data").resolve()
CLIP_META_TSV = DATA_PATH / "clip_embeddings_with_ids.tsv"
CLIP_EMB_NPY  = DATA_PATH / "clip_embeddings.npy"            # (N,768)

## strucutre here is:
#app_id	trace_id	image_path	e0	e1	e2	e


TEXT_META_TSV = DATA_PATH / "sbert_text_meta.tsv"
TEXT_EMB_NPY  = DATA_PATH / "sbert_text_embeddings.npy"      # (M,384)

## strucutre here is:
#screen_key	app_id	trace_id	screen_id	json_path

OUT_DIR = "notebook_outputs"
os.makedirs(OUT_DIR, exist_ok=True)
print("OUT_DIR:", OUT_DIR)


OUT_DIR: notebook_outputs


## Load metadata + embeddings
We normalize embeddings for cosine similarity.


In [12]:

# 3) Load CLIP
clip_meta = read_meta_tsv(CLIP_META_TSV, low_memory=False).copy()

# derive screen_id if missing
if "screen_id" not in clip_meta.columns:
    if "image_path" in clip_meta.columns:
        clip_meta["screen_id"] = clip_meta["image_path"].astype(str).apply(
            lambda p: os.path.splitext(os.path.basename(p))[0]
        )
    else:
        raise KeyError("CLIP meta TSV needs either screen_id or image_path.")

clip_meta = attach_screen_key(clip_meta)
clip_emb  = l2_normalize(read_embeddings_npy(CLIP_EMB_NPY))

print("CLIP meta rows:", len(clip_meta), "emb:", clip_emb.shape)

# 3b) Load TEXT
text_meta = read_meta_tsv(TEXT_META_TSV, low_memory=False).copy()
if "screen_key" not in text_meta.columns:
    text_meta = attach_screen_key(text_meta)

text_emb = l2_normalize(read_embeddings_npy(TEXT_EMB_NPY))
print("TEXT meta rows:", len(text_meta), "emb:", text_emb.shape)


CLIP meta rows: 66261 emb: (66261, 768)
TEXT meta rows: 66256 emb: (66256, 384)


In [36]:
## addtional debug:
#align_by_key(clip_meta, clip_emb, text_meta, text_emb)

print(clip_meta.columns)
print(text_meta.columns)
print(text_emb.shape)
print(clip_emb.max(), clip_emb.min())
print(text_emb.max(), text_emb.min())

Index(['app_id', 'trace_id', 'image_path', 'e0', 'e1', 'e2', 'e3', 'e4', 'e5',
       'e6',
       ...
       'e760', 'e761', 'e762', 'e763', 'e764', 'e765', 'e766', 'e767',
       'screen_id', 'screen_key'],
      dtype='object', length=773)
Index(['screen_key', 'app_id', 'trace_id', 'screen_id', 'json_path'], dtype='object')
(66256, 384)
0.60440385 -0.5640035
0.25245187 -0.24490902


## Align by screen_key
Keeps only the intersection of screens that have both modalities.


In [14]:

# 4) Align
meta, v, t = align_by_key(clip_meta, clip_emb, text_meta, text_emb)
print("Aligned screens:", len(meta))
print("v:", v.shape, "t:", t.shape)
meta.head()


Aligned screens: 66256
v: (66256, 768) t: (66256, 384)


,app_id,trace_id,image_path,e0,e1,e2,e3,e4,e5,e6,...,e760,e761,e762,e763,e764,e765,e766,e767,screen_id,screen_key
0,B4A.BigFivePersonalityTest,trace_0,../../dataset/traces/filtered_traces\B4A.BigFi...,0.021896,0.030844,0.000987,-0.021176,0.018079,0.020090,-0.025500,...,-0.038611,-0.003871,-0.018897,-0.005810,0.034046,0.019525,-0.016564,-0.007351,221,B4A.BigFivePersonalityTest::trace_0::221
1,CN.MyPrivateMessages,trace_0,../../dataset/traces/filtered_traces\CN.MyPriv...,0.017011,0.009498,-0.005017,-0.002625,0.008082,-0.003549,0.015750,...,-0.017555,0.001459,-0.010248,0.002893,-0.010426,0.015103,0.020255,-0.008485,15,CN.MyPrivateMessages::trace_0::15
2,DOCECG2.doctor,trace_0,../../dataset/traces/filtered_traces\DOCECG2.d...,0.015138,0.072915,0.007541,0.010914,0.019509,-0.015360,-0.017501,...,-0.020507,0.008312,0.008881,-0.025093,0.022912,-0.014049,-0.039133,-0.035601,58,DOCECG2.doctor::trace_0::58
3,Gecko.Droid.PhysicsHelper,trace_0,../../dataset/traces/filtered_traces\Gecko.Dro...,0.043462,0.058752,0.018171,-0.020830,-0.040667,0.037265,-0.016190,...,-0.004884,-0.039759,-0.008874,0.003998,0.016603,0.017640,-0.006107,0.038262,194,Gecko.Droid.PhysicsHelper::trace_0::194
4,Gecko.Droid.PhysicsHelper,trace_0,../../dataset/traces/filtered_traces\Gecko.Dro...,0.047438,0.023143,0.041957,-0.021759,-0.025964,0.028507,0.021954,...,-0.010608,-0.004325,-0.024038,-0.016243,0.062699,0.010949,-0.016785,0.001056,269,Gecko.Droid.PhysicsHelper::trace_0::269


## Fuse (concat)
Adjust weights to emphasize visual vs text.


In [ ]:

# 5) Fuse
ALPHA = 1.0
BETA  = 1.0
fused = concat_fuse(v, t, alpha=ALPHA, beta=BETA, normalize=True)
print("Fused:", fused.shape)


## kNN sanity check (top-5)
Blockwise neighbor search (no NxN matrix).


In [ ]:

# 6) Neighbors
import random

def print_neighbors(X, idx, label, k=5, block=8192):
    nbrs = topk_cosine_neighbors_blockwise(X, idx, k=k, block=block)
    print(f"{label} neighbors:")
    for j, s in nbrs:
        print(f"  {meta.iloc[j]['screen_key']}  sim={s:.4f}")

random.seed(42)
picks = random.sample(range(len(meta)), k=min(5, len(meta)))

for qi in picks:
    print("\n---")
    print("QUERY:", meta.iloc[qi]["screen_key"])
    print_neighbors(v, qi, "CLIP")
    print_neighbors(t, qi, "TEXT")
    print_neighbors(fused, qi, "FUSED(concat)")


## Clustering (MiniBatchKMeans baseline)
Try multiple K values and compare cluster size stats.


In [ ]:

# 7) Clustering
K = 80
cluster_id = kmeans_cluster(fused, k=K, seed=42)

clusters = meta[["screen_key","app_id","trace_id","screen_id"]].copy()
clusters["cluster_id"] = cluster_id

out_path = os.path.join(OUT_DIR, f"screen_clusters_k{K}.tsv")
clusters.to_csv(out_path, sep="\t", index=False)

sizes = clusters["cluster_id"].value_counts()
print("Saved:", out_path)
print("Cluster sizes: min=", int(sizes.min()), "median=", int(sizes.median()), "max=", int(sizes.max()), "num_clusters=", sizes.shape[0])
sizes.head()


## Near-duplicate scan (top-1 similarity)
Computes each screen's best neighbor similarity (excluding itself). Subsample while debugging.


In [ ]:

# 8) Near-duplicate scan
def top1_neighbor_scan(X, block_q=1024, block_db=8192):
    # Returns best_sim (N,), best_j (N,). X must be L2-normalized.
    N, D = X.shape
    best_sim = np.full(N, -np.inf, dtype=np.float32)
    best_j   = np.full(N, -1, dtype=np.int32)

    for qs in range(0, N, block_q):
        qe = min(qs + block_q, N)
        Q = X[qs:qe]

        block_best_sim = np.full(qe-qs, -np.inf, dtype=np.float32)
        block_best_j   = np.full(qe-qs, -1, dtype=np.int32)

        for ds in range(0, N, block_db):
            de = min(ds + block_db, N)
            B = X[ds:de]
            sims = Q @ B.T

            # exclude self on overlap
            for i in range(qs, qe):
                if ds <= i < de:
                    sims[i-qs, i-ds] = -np.inf

            j_local = np.argmax(sims, axis=1)
            s_local = sims[np.arange(qe-qs), j_local]

            better = s_local > block_best_sim
            block_best_sim[better] = s_local[better]
            block_best_j[better]   = (ds + j_local[better]).astype(np.int32)

        best_sim[qs:qe] = block_best_sim
        best_j[qs:qe]   = block_best_j

    return best_sim, best_j

THRESH = 0.99995

# Optional subsample for speed:
# idx = np.random.default_rng(42).choice(len(fused), size=20000, replace=False)
# X_scan = fused[idx]
# meta_scan = meta.iloc[idx].reset_index(drop=True)

X_scan = fused
meta_scan = meta.reset_index(drop=True)

best_sim, best_j = top1_neighbor_scan(X_scan)

df_dup = pd.DataFrame({
    "screen_key": meta_scan["screen_key"].values,
    "top1_neighbor_key": meta_scan.iloc[best_j]["screen_key"].values,
    "top1_sim": best_sim
})

hits = df_dup[df_dup["top1_sim"] >= THRESH].sort_values("top1_sim", ascending=False)

out_path = os.path.join(OUT_DIR, f"near_duplicates_top1_thresh{THRESH}.tsv")
hits.to_csv(out_path, sep="\t", index=False)

print("Threshold:", THRESH)
print("Count >= threshold:", len(hits), "/", len(df_dup))
print("Saved:", out_path)
hits.head(15)


## Optional: trace → cluster sequences & transitions
Uses `uiflow.analysis.transitions`.


In [ ]:

# 9) Transition model stub
from uiflow.analysis.transitions import build_trace_sequences, bigram_counts, bigram_next_distribution

seqs = build_trace_sequences(meta, cluster_id)
counts = bigram_counts(seqs)
P = bigram_next_distribution(counts, num_clusters=K, smoothing=1.0)

print("Num traces:", len(seqs))
print("Transition matrix:", P.shape)
